# Load dim_region

dim_region is derived from bronze.products data.

Starting the notebook with %run "/Workspace/Shared/notebook_init"  loads common variables and constants used across all notebooks for the Vinoworld project

CATALOG, BRONZE, SILVER, GOLD, AUDIT, RAW_FILES,
PIPELINE_RUN_ID, Utils, F, Row, datetime etc.

import uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, input_file_name, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType
sys.path.append("/Workspace/Shared")

import pipeline_utils as Utils
from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert


CLAUDE deleted the cell that populated dim_region when it deleted the %skip cells.  Had to add it back in


In [0]:
%run "/Workspace/Shared/notebook_init"

In [0]:
%skip

import sys
sys.path.append("/Workspace/Shared")

import  time
from datetime import datetime, timezone
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DoubleType, IntegerType, BooleanType
)


from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert
# from pipeline_utils import get_notebook_context, capture_exception, move_all_files
import pipeline_utils as Utils


CATALOG   = "Vinoworld"
SILVER    = f"{CATALOG}.silver"
AUDIT     = f"{CATALOG}.audit"
RAW_FILES = "/Volumes/vinoworld/datafiles/"


# --------------------------------------------------------------------------
# Get pipeline parameters for pipeline_log table info.
# --------------------------------------------------------------------------

LOCAL_PIPELINE_ID      = 22222     # Use int(time.time() * 1000)   to generate



# Pipeline-level identifiers — shared across all notebooks in a pipeline run.
# Read from parent via widgets; fall back to standalone mode if not provided.
dbutils.widgets.text("pipeline_run_id", "")
value = dbutils.widgets.get("pipeline_run_id")

try:
    PIPELINE_RUN_ID = int(value)
except ValueError:
    PIPELINE_RUN_ID = LOCAL_PIPELINE_ID
    #raise ValueError(f"Invalid pipeline_run_id: {value}")

except Exception:
    # Not running in Databricks (e.g. local dev) — use safe defaults
    PIPELINE_RUN_ID   = LOCAL_PIPELINE_ID



In [0]:
# Imports and constants specific to the Arancione Bronze load.
# STORE_NAME identifies the source store; SOURCE_SUBPATH is the subfolder
# under RAW_FILES; TARGET_TABLE is the fully-qualified Bronze table name.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone
import traceback

%load_ext autoreload
%autoreload 2

SOURCE_SUBPATH = "masterdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{SILVER}.dim_region"

# print(f"Products Source Path:  {SOURCE_PATH}")


In [0]:
# ----------------------------------------------------------------------
# Setup the variables need for initial call to pipeline_step_log_upsert
# ----------------------------------------------------------------------

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_name']

#logger.info(f"Inserting pipeline_step_log record for notebook {notebook_name}")

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = TARGET_TABLE
status            = "running"
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)

# All Parameters
# pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table, rows_read, rows_written,  ended_timestamp,  error_message)

In [0]:
## ============================================================
#  Pipeline: vinoworld.bronze.products → vinoworld.silver.dim_product
#  Pattern: Type 2 SCD via multi-statement approach
#  Databricks Delta Lake (Free / Standard tier)
#  ============================================================

#  ────────────────────────────────────────────────────────────
#  STEP 1: Stage incoming data — cast, filter, deduplicate,
#          and compute the RowHash for change detection.
# ────────────────────────────────────────────────────────────

try:
    # Defining the SQL logic using the optimized QUALIFY clause for idempotency
    sql_query = """
         MERGE INTO vinoworld.silver.dim_region a
        USING (
            SELECT DISTINCT
                p.province       AS Province,
                p.region_1       AS RegionName,
                p.region_2       AS SubRegionName,
                current_timestamp() AS InsertedDate,
                current_timestamp() AS UpdatedDate
            FROM vinoworld.bronze.products p
            WHERE p.province IS NOT NULL
        ) t
        ON  a.Province = t.Province
        AND a.RegionName <=> t.RegionName
        AND a.SubRegionName <=> t.SubRegionName

        WHEN NOT MATCHED THEN
            INSERT (
                Province,
                RegionName,
                SubRegionName,
                InsertedDate,
                UpdatedDate
            )
            VALUES (
                t.Province,
                t.RegionName,
                t.SubRegionName,
                t.InsertedDate,
                t.UpdatedDate
            )
    """

    # Execute the query
    spark.sql(sql_query)
    print("Success: dim_region was loaded.")


except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise

In [0]:
%skip


metrics = spark.sql(f"DESCRIBE HISTORY vinoworld.silver.dim_product LIMIT 1") \
               .select("operationMetrics") \
               .collect()[0][0]

rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
rows_updated  = int(metrics.get("numTargetRowsUpdated",  0))
rows_deleted  = int(metrics.get("numTargetRowsDeleted",  0))

print(f" Rows  Inserted: {rows_inserted:,}")
print(f" Rows     Updated: {rows_updated:,}"   )
